# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
import re

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
reg_model = ''
runs = '-1-'
comment = "acute_all-vary_b_I"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
#sec_diff_bias_list = []
cell_series_list = []
# lineage_diff_nets = []

In [3]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/'
run_list = [int(re.search('sim_batch_(.*?)\.', f).group(1)) for f in os.listdir(d_rerun) if 'sim_batch' in f and comment in f and runs in f]
out = [str(x) for x in [k for k in np.arange(0, 498)] if x not in run_list]
print(len(out))
print(' '.join((out)))

121
0 2 180 181 355 356 499 500 501 502 503 504 505 506 507 508 509 510 511 512 513 514 515 516 517 518 519 520 521 522 523 524 525 526 527 528 529 530 531 532 533 534 535 536 537 538 539 540 541 542 543 544 545 546 547 548 549 550 551 552 553 554 555 556 557 558 559 560 561 562 563 564 565 566 567 568 569 570 571 572 573 574 575 576 577 578 579 580 581 582 583 584 585 586 587 588 589 590 591 592 593 594 595 596 597 598 599 600 601 602 603 604 605 606 607 608 609 610 611 612 613


In [4]:
num_cpu = 100
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and comment in f and 'sim_batch' in f]
num_files = len(file_list)

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    parameters = np.array(import_dict["parameters"])
    sim_sum = np.array(import_dict["summary_stats"])

    out = np.hstack((parameters, sim_sum))

    return out

# create dataframe of infection response statistics
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
# mean_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
#                                                                                                         for file_name in file_list)), 
#                        columns = [i for i in var_names]).groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean()
full_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
                                                                                                        for file_name in file_list)), 
                       columns = [i for i in var_names])

# # Save datasets
full_df.to_pickle(os.path.join(d, "raw", "stacked_full_data"+runs+"runs"+'-'+comment)+'.pkl')

In [5]:
with pd.option_context('display.max_columns', None):
    display(full_df)

,S_0,I_0,b_I,d_S,d_I,d_IE,K_I,d_H,K_H,N_0,max_Na,t_bind,t_unbind,t_Na_div,t_E_div,t_M_div,t_E_die,t_cycle,psi_myc_I,psi_myc_HI,psi_myc_HE,L0_Na,psi_NE_I,psi_NE_HI,psi_NE_HE,L0_NE,psi_EM_I,psi_EM_HI,psi_EM_HE,L0_EM,psi_Edie_I,psi_Edie_HI,psi_Edie_HE,L0_Edie,p_load,T_max_pI,T_min_pI,harm_pI,harm_pS,max_pE,T_pE_max,T_pE_start,max_eM,T_pEcyteM,T_pE_end,frac_cM,int_pHE,int_pHI,E_end
0,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.6,1.2,0.8,-0.6,-1.6,1.2,0.8,0.0,-0.0,0.0,0.0,-27.0,1.6,-1.2,-0.8,-2.4,5.684342e-17,0.00,0.02,1.010000e+03,9.971513e+06,619100.0,11.11,3.51,0.0,0.0,30.00,0.514851,2.186589e+06,2.533802e+02,11818.0
1,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.6,1.2,0.8,-0.6,-1.6,1.2,0.8,0.0,-0.0,0.0,0.0,-27.0,1.6,-1.2,-0.8,-1.8,-5.684342e-17,0.00,0.02,1.010000e+03,9.761992e+06,509353.0,13.47,4.18,0.0,0.0,30.00,0.489130,1.877727e+06,2.533063e+02,3017.0
2,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.6,1.2,0.8,-0.6,-1.6,1.2,0.8,0.0,-0.0,0.0,0.0,-27.0,1.6,-1.2,-0.8,-1.2,0.000000e+00,0.00,0.02,1.010000e+03,8.752316e+06,392082.0,17.67,4.78,0.0,0.0,30.00,0.551181,1.535347e+06,2.533711e+02,3439.0
3,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.6,1.2,0.8,-0.6,-1.6,1.2,0.8,0.0,-0.0,0.0,0.0,-27.0,1.6,-1.2,-0.8,-0.6,2.273179e-08,0.00,0.02,1.010000e+03,1.291592e+04,141.0,15.31,8.08,0.0,0.0,23.54,0.505952,7.731082e+02,2.530676e+02,65.0
4,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.6,1.2,0.8,-0.6,-1.6,1.2,0.8,0.0,-0.0,0.0,0.0,-27.0,1.6,-1.2,-0.8,0.0,2.572250e-07,0.00,0.02,1.010000e+03,7.279660e+02,22.0,4.84,30.00,0.0,0.0,0.00,0.479730,7.603787e+01,2.532908e+02,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141768220,10000000.0,1000.0,2.500000e-07,0.01,0.5,12.0,10000000.0,1.0,100000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.8,-1.2,0.8,1.8,-0.8,-1.2,0.8,-3.0,-0.0,-0.0,0.0,-27.0,0.8,1.2,-0.8,1.2,4.598407e-02,5.42,23.86,9.932583e+06,1.455100e+01,33.0,24.08,4.09,0.0,0.0,0.00,0.969773,3.160008e+00,1.987816e+06,14.0
141768221,10000000.0,1000.0,2.500000e-07,0.01,0.5,12.0,10000000.0,1.0,100000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.8,-1.2,0.8,1.8,-0.8,-1.2,0.8,-3.0,-0.0,-0.0,0.0,-27.0,0.8,1.2,-0.8,1.8,4.599013e-02,5.42,23.86,9.932571e+06,1.727006e+01,3.0,3.29,3.96,0.0,0.0,0.00,0.971576,9.725264e+00,1.987812e+06,0.0
141768222,10000000.0,1000.0,2.500000e-07,0.01,0.5,12.0,10000000.0,1.0,100000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.8,-1.2,0.8,1.8,-0.8,-1.2,0.8,-3.0,-0.0,-0.0,0.0,-27.0,0.8,1.2,-0.8,2.4,4.598992e-02,5.42,23.86,9.932588e+06,1.375372e+00,6.0,20.52,4.17,0.0,0.0,0.00,0.981912,9.527559e-01,1.987818e+06,0.0
141768223,10000000.0,1000.0,2.500000e-07,0.01,0.5,12.0,10000000.0,1.0,100000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.8,-1.2,0.8,1.8,-0.8,-1.2,0.8,-3.0,-0.0,-0.0,0.0,-27.0,0.8,1.2,-0.8,3.0,4.598985e-02,5.42,23.86,9.932584e+06,5.373237e+00,5.0,13.08,4.70,0.0,0.0,0.00,0.981723,3.737232e+00,1.987816e+06,0.0


In [6]:
# Create additional variables
virs = np.unique(full_df[['I_0','d_I','K_I','b_I','K_H','N_0']].values, axis = 0)

full_df['antigenicity_over_harm'] = antigenicity_over_harm(full_df)
full_df['stim_pI'] = np.log(1 + (full_df['p_load']/full_df['K_I']))
full_df['stim_pHI'] = np.log(1 + (full_df['int_pHI']/full_df['K_H']))
full_df['stim_pHE'] = np.log(1 + (full_df['int_pHE']/full_df['K_H']))
#full_df['scaled_min_pS'] = full_df['min_pS']/full_df['S_0']

# identify Biologically evidenced networks
keep_vars = ['harm_pI', 'harm_pS', 'frac_cM', 'max_pE',
             'T_pE_start', 'T_pE_max', 'T_pE_end',
             'stim_pI', 'stim_pHI', 'stim_pHE',
             'E_end', 'antigenicity_over_harm']

In [7]:
# save data sets
full_infection_scenarios = []
mean_of_infection_scenarios = []
std_of_infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_0

for l, (I_0, d_I, K_I, b_I, K_H, N_0) in enumerate(tqdm(virs)):
    data = full_df.loc[(full_df["d_I"] == d_I)*(full_df["K_I"] == K_I)*(full_df["b_I"] == b_I)*(full_df["K_H"] == K_H)*(full_df["N_0"] == N_0)*(full_df["I_0"] == I_0), 
    ['b_I','d_I', 'K_I', 'I_0','S_0', 'N_0', 'd_S', 'K_H'] + Na_reg + NE_reg + EM_reg + EE_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_I = K_I, d_I = d_I, b_I = b_I,
                                   infection_model = "cancer" if b_I >= b_C else "acute")
    no_eff_stats = no_eff_data[l]["summary_stats"]

    data.loc[:,"harm_pI_noprotection"] = no_eff_stats[3]/S_0
    data.loc[:,"peff_clearance"] = (no_eff_stats[3] - data['harm_pI'].to_numpy())/S_0
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/S_0
    data.loc[:,"peff_protection"] = data['peff_clearance'] - data['peff_toxicity']
    data.loc[:,"peff_scaled_protection"] = data['peff_protection']/data['harm_pI_noprotection']

    mean_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean())
    std_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).std())
    full_infection_scenarios.append(data)

# stack datasets
pd.concat(mean_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(std_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(full_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/processed_full_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(mean_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(std_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/list_processed_full_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(full_infection_scenarios, f)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 81/81 [20:43<00:00, 15.35s/it]


In [8]:
# Clear memory
del full_df, mean_of_infection_scenarios, std_of_infection_scenarios, full_infection_scenarios